In [1]:
import math
import os
from pathlib import Path

import h3
import numpy as np
import polars as pl
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm
import h3pandas

2024-10-30 18:08:40.152197: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-30 18:08:40.152256: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-30 18:08:40.172680: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-30 18:08:40.222428: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-30 18:08:40.958588: W tensorflow/compiler/tf2

In [2]:
params = {
    "export_dir": "/data-ssd/alex/datasets/vision-export-20240929050006-aka-2.17",
    "export_short_version": "2.17",
    "train_only_cid_data": True,
    "train_only_wild_data": False,
    "h3_resolution": 6,
    "num_random_samples": 100_000,
    "elevation_file": "/home/alex/elevation_h3_resolution6.csv",
    "experiment_dir": "/data-ssd/alex/experiments/geo_prior_tf/2_17_inner",
    "batch_size": 1024,
    "num_epochs": 200,
    "initial_lr": 0.0005,
    "shuffle_buffer_size": 50_000,
    "full_shuffle_before_tfrecords": False,
    "lr_warmup_cosine_decay": True,
    "wandb_project": "geomodel_tf",
}

In [3]:
h3_column_name = "h3_0{}".format(params["h3_resolution"])

In [4]:
export_dir = Path(params["export_dir"])

In [5]:
spatial_data_polars = pl.read_parquet(
    export_dir / "spatial_data.parquet",
    columns=["latitude", "longitude", "spatial_class_id", "community", "captive"]
)
len(spatial_data_polars)

216897992

In [53]:
tax_polars = pl.read_csv(
    export_dir / "taxonomy.csv",
    schema_overrides={
        "rank_level": float,
    }
)
len(tax_polars)

141319

### clean dataset - remove inner nodes

In [7]:
# skip this stuff - we are training at inner nodes
# spatial_data_polars = spatial_data_polars.filter(
#     pl.col("spatial_class_id").is_in(leaf_tax_polars["spatial_class_id"])
# )
len(spatial_data_polars)

216897992

### clean dataset - drop non-cid data

In [8]:
spatial_data_polars = spatial_data_polars.filter(
    pl.col("community") == 1
)
len(spatial_data_polars)

154403161

#### clean dataset - rename

In [9]:
spatial_data_polars = spatial_data_polars.rename({
    "latitude": "lat",
    "longitude": "lng",
})

#### clean dataset - drop invalid locations

In [10]:
# drop invalid locations
spatial_data_polars = spatial_data_polars.filter([
    pl.col("lat") <= 90,
    pl.col("lat") >= -90,
    pl.col("lng") <= 180,
    pl.col("lng") >= -180,
])
len(spatial_data_polars)

154403154

#### clean dataset - drop null island locations

In [11]:
spatial_data_polars = spatial_data_polars.filter([
    pl.col("lat") != 0,
    pl.col("lng") != 0,
])
len(spatial_data_polars)

154402427

In [12]:
#spatial_data_polars = spatial_data_polars[["lat", "lng", "spatial_class_id"]]

# let's expand and include data at inner nodes

In [54]:
taxon_children = tax_polars.select(pl.col("parent_taxon_id"), pl.col("taxon_id")).group_by(
    pl.col("parent_taxon_id")
).all()

In [55]:
taxon_children.filter(pl.col("parent_taxon_id") == 1)["taxon_id"].to_list()

[[2,
  47115,
  47120,
  47491,
  47534,
  47549,
  48051,
  48824,
  51280,
  51508,
  51836,
  52319,
  54960,
  63142,
  68104,
  68235,
  122158,
  124337,
  126917,
  151827,
  151829,
  151831,
  211191]]

In [56]:
taxon_children = taxon_children.rename({
    "parent_taxon_id": "taxon_id",
    "taxon_id": "child_taxon_ids"
})

In [57]:
tax_polars = tax_polars.join(
    taxon_children,
    on="taxon_id",
    how="left"
)

In [59]:
tax_polars.sample(5)

parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name,child_taxon_ids
i64,i64,f64,i64,i64,i64,str,list[i64]
959951,199869,20.0,null,null,140229,"""Zographetus""",[494429]
67720,343241,10.0,83557,1,83557,"""Asphondylia auripila""",null
1303561,56074,30.0,null,null,122540,"""Callionymidae""","[56075, 87174, 87830]"
1558546,154366,10.0,58812,3,58812,"""Amanita ceciliae""",null
578061,199449,10.0,39735,1,39735,"""Amegilla chlorocyanea""",null


In [98]:
def descendents_of_taxon_id(tid):
    taxon = tax_polars.filter(
        pl.col("taxon_id") == tid
    )

    if taxon["child_taxon_ids"].item() is None:
        # base case, return taxon id
        return [taxon["taxon_id"].item()]
    else:
        # recursive case
        children = taxon["child_taxon_ids"].to_list()[0]
        descs = []
        for child in children:
            descs += descendents_of_taxon_id(child)
        return descs


In [99]:
all_taxon_ids = tax_polars["taxon_id"].to_list()
np.random.shuffle(all_taxon_ids)

In [ ]:
descendents_dict = dict()
for tid in tqdm(all_taxon_ids):
    descendents = descendents_of_taxon_id(tid)
    descendents_dict[tid] = descendents

  0%|          | 0/141319 [00:00<?, ?it/s]

In [94]:
tax_polars.filter(pl.col("taxon_id") == 788694)

parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name,child_taxon_ids
i64,i64,f64,i64,i64,i64,str,list[i64]
85594,788694,27.0,null,null,113545,"""Apogoninae""","[86562, 86563, … 500371]"


In [95]:
tax_polars.filter(pl.col("taxon_id") == 86562)

parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name,child_taxon_ids
i64,i64,f64,i64,i64,i64,str,list[i64]
788694,86562,20.0,null,null,111928,"""Apogon""","[94394, 118514, … 131488]"


In [97]:
131488 in descendents_dict[788694] 

True

# ok let's do h3 stuff

In [ ]:
spatial_data_polars_h3 = spatial_data_polars.with_columns(
    pl.struct("lat", "lng")
    .map_elements(
        lambda x: h3.geo_to_h3(x["lat"], x["lng"], params["h3_resolution"]),
        return_dtype=str
    )
    .alias(h3_column_name)
)
len(spatial_data_polars_h3)

In [ ]:
spatial_data_pandas_h3 = spatial_data_pandas.h3.geo_to_h3(params["h3_resolution"])
len(spatial_data_pandas_h3)

In [ ]:
spatial_data_polars_h3.head(5)

In [ ]:
spatial_data_pandas_h3 = spatial_data_pandas_h3.reset_index()

In [ ]:
spatial_data_pandas_h3.head(5)

In [ ]:
spatial_data_polars_h3.head(5).to_pandas()[["h3_06", "lat", "lng", "spatial_class_id"]] == spatial_data_pandas_h3.head(5)

In [ ]:
(spatial_data_polars_h3.to_pandas()[["h3_06", "lat", "lng", "spatial_class_id"]] == spatial_data_pandas_h3).all().all()

In [ ]:
h3_column_name

In [ ]:
spatial_data_polars_h3_dense = (
    spatial_data_polars_h3
    .group_by(h3_column_name)
    .agg(
        pl.col("spatial_class_id")
        .unique()
    )
)

In [ ]:
spatial_data_polars_h3_dense.sort("h3_06").tail(5)

In [ ]:
spatial_data_pandas_h3_dense = spatial_data_pandas_h3.pivot_table(
    index=h3_column_name, values="spatial_class_id", aggfunc=set
)

In [ ]:
spatial_data_pandas_h3_dense.tail(5)

In [ ]:
spatial_data_pandas_h3_dense[
    "spatial_class_id"
] = spatial_data_pandas_h3_dense.spatial_class_id.apply(lambda x: list(x))


In [ ]:
len(spatial_data_pandas_h3_dense)

In [ ]:
len(spatial_data_polars_h3_dense)

In [ ]:
spatial_data_pandas_h3_dense.sample(5).sort_index()

In [ ]:
spatial_data_polars_h3_dense.filter(
    pl.col("h3_06").is_in([
        "86118a4c7ffffff",
        "862c2a227ffffff",
        "862e69a77ffffff",
        "866d756dfffffff",
        "86b241d77ffffff",
    ])
).sort("h3_06")

looks good so far

# random samples for negatives

In [ ]:
def make_samples(batch_size):
    rand_loc = np.random.uniform(size=(batch_size, 2))

    theta1 = 2.0 * math.pi * rand_loc[:, 0]
    theta2 = np.arccos(2.0 * rand_loc[:, 1] - 1.0)

    lat = 1.0 - 2.0 * theta2 / math.pi
    lng = (theta1 / math.pi) - 1.0

    return list(zip(lng, lat))

In [ ]:
samples = make_samples(params["num_random_samples"])
scaled_samples = [(x * 180, y * 90) for x, y in samples]
# scaled_samples = list(zip(*scaled_samples))

In [ ]:
negatives_df_polars = pl.DataFrame(
    list(zip(*scaled_samples)),
    ["lng", "lat"]
)

In [ ]:
negatives_df_pandas = pd.DataFrame(
    scaled_samples, 
    columns=["lng", "lat"]
)

In [ ]:
(negatives_df_polars.to_pandas() == negatives_df_pandas).all().all()

In [ ]:
negatives_polars_h3 = negatives_df_polars.with_columns(
    pl.struct("lat", "lng")
    .map_elements(
        lambda x: h3.geo_to_h3(x["lat"], x["lng"], params["h3_resolution"]),
        return_dtype=str
    )
    .alias(h3_column_name)
)
negatives_polars_h3.head(5)

In [ ]:
negatives_pandas_h3 = negatives_df_pandas.h3.geo_to_h3(params["h3_resolution"])
negatives_pandas_h3 = negatives_pandas_h3.reset_index()[["lng", "lat", "h3_06"]]

In [ ]:
(negatives_polars_h3.to_pandas() == negatives_pandas_h3).all().all()

In [ ]:
negatives_pandas_h3 = negatives_pandas_h3.drop_duplicates("h3_06", keep="first")
negatives_pandas_h3 = negatives_pandas_h3.reset_index()[["lng", "lat", "h3_06"]]

In [ ]:
negatives_polars_h3 = negatives_polars_h3.unique(subset="h3_06", keep="first")

In [ ]:
len(negatives_pandas_h3), len(negatives_polars_h3)

In [ ]:
negatives_polars_h3 = negatives_polars_h3.sort("h3_06")
negatives_pandas_h3 = negatives_pandas_h3.sort_values("h3_06")
negatives_pandas_h3 = negatives_pandas_h3.reset_index()[["lng", "lat", "h3_06"]]

In [ ]:
( negatives_pandas_h3 == negatives_polars_h3.to_pandas() ).all().all()

### merge negatives/empties with spatial data

In [ ]:
negatives_pandas_h3 = negatives_pandas_h3[
    ~negatives_pandas_h3.h3_06.isin(spatial_data_pandas_h3_dense.index)
]
negatives_pandas_h3 = negatives_pandas_h3.sort_values("h3_06")
negatives_pandas_h3 = negatives_pandas_h3.reset_index()[["lng", "lat", "h3_06"]]
negatives_pandas_h3.head(5)

In [ ]:
len(negatives_pandas_h3)

In [ ]:
# trim any negatives that would duplicate positives
negatives_polars_h3 = negatives_polars_h3.filter(
    ~pl.col(h3_column_name).is_in(spatial_data_polars_h3_dense[h3_column_name])
)
negatives_polars_h3.head(), len(negatives_polars_h3)

In [ ]:
polars_neg_h3_indices = set(negatives_polars_h3["h3_06"].to_list())
len(polars_neg_h3_indices)

In [ ]:
pandas_neg_h3_indices = set(negatives_pandas_h3["h3_06"].to_list())
len(pandas_neg_h3_indices)

In [ ]:
len(pandas_neg_h3_indices.difference(polars_neg_h3_indices))

In [ ]:
len(pandas_neg_h3_indices.intersection(polars_neg_h3_indices))

In [ ]:
(negatives_polars_h3.to_pandas() == negatives_pandas_h3).all().all()

#### add empty spatial class id column

In [ ]:
negatives_pandas_h3["spatial_class_id"] = negatives_pandas_h3["lat"].apply(
    lambda _: []
)
negatives_pandas_h3

In [ ]:
negatives_polars_h3 = negatives_polars_h3.with_columns(
    pl.lit([], dtype=pl.datatypes.List(pl.datatypes.Int64))
    .alias("spatial_class_id")
)

In [ ]:
negatives_polars_h3.to_pandas()

In [ ]:
negatives_pandas_h3

In [ ]:
negatives_pandas_h3.dtypes

In [ ]:
negatives_polars_h3.to_pandas().dtypes

In [ ]:
(negatives_polars_h3.to_pandas() == negatives_pandas_h3)

In [ ]:
negatives_polars_h3 = negatives_polars_h3[[h3_column_name, "spatial_class_id"]]
negatives_polars_h3.head(3)

In [ ]:
negatives_pandas_h3 = negatives_pandas_h3[[h3_column_name, "spatial_class_id"]]
negatives_pandas_h3.head(3)

In [ ]:
combined_polars = pl.concat([
    spatial_data_polars_h3_dense,
    negatives_polars_h3,
]).sort("h3_06")

In [ ]:
combined_pandas = pd.concat([negatives_pandas_h3, spatial_data_pandas_h3_dense.reset_index()])
combined_pandas = combined_pandas.sort_values("h3_06")
combined_pandas = combined_pandas.reset_index()[["h3_06", "spatial_class_id"]]

# combined["spatial_class_id"] = combined["spatial_class_id"].apply(
#     lambda d: d if isinstance(d, list) else []
# )
# combined = combined[["spatial_class_id"]]
# combined = combined.h3.h3_to_geo()
# combined = combined.sort_index()
# combined["lng"] = combined["geometry"].x
# combined["lat"] = combined["geometry"].y
# _ = combined.pop("geometry")


In [ ]:
combined_pandas.tail(5)

In [ ]:
combined_polars.to_pandas().tail(5)

# add elevation

In [ ]:
elevation_pandas = pd.read_csv(
    params["elevation_file"]
)
elevation_pandas.tail()

In [ ]:
elevation_polars = pl.read_csv(
    params["elevation_file"]
)
elevation_polars.to_pandas().tail()

In [ ]:
elevation_pandas["elevation"].to_numpy()

In [ ]:
elevation_polars["elevation"].to_numpy()

In [ ]:
equality = elevation_pandas["elevation"].to_numpy() == elevation_polars["elevation"].to_numpy()

In [ ]:
np.isclose(
    elevation_pandas["elevation"].to_numpy(),
    elevation_polars["elevation"].to_numpy()
).all()
# rounding errors? weird

In [ ]:
elevation_polars = elevation_polars.with_columns(
    pl.when(pl.col("elevation") > 0)
    .then(pl.col("elevation") / pl.col("elevation").max())
    .otherwise(pl.col("elevation") / pl.col("elevation").min() * -1)
)

In [ ]:
# elevation_pandas.loc[elevation_pandas.elevation > 0, "elevation"] = elevation_pandas.loc[elevation_pandas.elevation > 0, "elevation"] / np.max(elevation_pandas.elevation)
# elevation_pandas.loc[elevation_pandas.elevation < 0, "elevation"] = elevation_pandas.loc[elevation_pandas.elevation < 0, "elevation"] / np.min(elevation_pandas.elevation)

In [ ]:
max_elev = elevation_pandas.elevation.max()
min_elev = elevation_pandas.elevation.min()

In [ ]:
elevation_pandas[:100]

In [ ]:
elevation_pandas["elevation_normalized"] = elevation_pandas.elevation.apply(
    lambda x: x / max_elev if x > 0 else -1 * (x / min_elev)
)

In [ ]:
elevation_pandas

In [ ]:
combined_pandas

In [ ]:
combined_with_elevation_pandas = pd.merge(
    combined_pandas,
    elevation_pandas,
    left_on="h3_06",
    right_on="h3_06",
)
_ = combined_with_elevation_pandas.pop("elevation")
combined_with_elevation_pandas = combined_with_elevation_pandas.rename({
    "elevation_normalized": "elevation"
}, axis="columns")
combined_with_elevation_pandas.sample(5).sort_values("h3_06")

In [ ]:
combined_with_elevation_polars = combined_polars.join(
    elevation_polars,
    on="h3_06"
)

In [ ]:
combined_with_elevation_polars.filter(
    pl.col("h3_06").is_in([
        "8611817b7ffffff",
        "8611b4cc7ffffff",
        "86182a6d7ffffff",
        "862a166cfffffff",
        "865234587ffffff",
    ])
).sort("h3_06")

# convert to geo and normalize

In [ ]:
combined_with_elevation_pandas = combined_with_elevation_pandas.set_index("h3_06").h3.h3_to_geo()
combined_with_elevation_pandas = combined_with_elevation_pandas.sort_index()
combined_with_elevation_pandas["lng"] = combined_with_elevation_pandas["geometry"].x
combined_with_elevation_pandas["lat"] = combined_with_elevation_pandas["geometry"].y
_ = combined_with_elevation_pandas.pop("geometry")

In [ ]:
combined_with_elevation_pandas.sample(3)

In [ ]:
combined_with_elevation_polars = combined_with_elevation_polars.with_columns(
    pl.col(h3_column_name)
    .map_elements(lambda x: h3.h3_to_geo(x))
    .alias("xy"),
)
combined_with_elevation_polars = combined_with_elevation_polars.with_columns([
    pl.col("xy")
    .map_elements(lambda x: x[0], return_dtype=float)
    .alias("lat"),
    pl.col("xy")
    .map_elements(lambda x: x[1], return_dtype=float)
    .alias("lng")
])

In [ ]:
combined_with_elevation_polars.filter(
    pl.col("h3_06").is_in([
        "86c2e2a2fffffff",
        "8631aaa8fffffff",
        "863f35457ffffff",
    ])
)

In [ ]:
combined_with_elevation_polars = combined_with_elevation_polars.with_columns([
    (pl.col("lng") / 180.0).alias("norm_lng"),
    (pl.col("lat") / 90.0).alias("norm_lat"),
])

In [ ]:
combined_with_elevation_polars = combined_with_elevation_polars.with_columns([
    (pl.col("norm_lng") * math.pi).sin().alias("a"),
    (pl.col("norm_lat") * math.pi).sin().alias("b"),
    (pl.col("norm_lng") * math.pi).cos().alias("c"),
    (pl.col("norm_lat") * math.pi).cos().alias("d"),
])

In [ ]:
combined_with_elevation_polars.sample(5)

# make tfrecords

In [ ]:
len(combined_with_elevation_pandas)

In [ ]:
def create_tf_example(l0, l1, l2, l3, elevation, leaf_class_ids):
    tf_example = tf.train.Example(
        features=tf.train.Features(
            feature={
                "l0": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[l0])
                ),
                "l1": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[l1])
                ),
                "l2": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[l2])
                ),
                "l3": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[l3])
                ),
                "elevation": tf.train.Feature(
                    float_list=tf.train.FloatList(value=[elevation])
                ),
                "leaf_class_ids": tf.train.Feature(
                    int64_list=tf.train.Int64List(value=leaf_class_ids)
                ),
            }
        )
    )
    return tf_example


In [ ]:
os.makedirs(
    os.path.join(params["export_dir"], "geo_spatial_grid_datasets"),
    exist_ok=True,
)
tfrecord_file = os.path.join(
    params["export_dir"],
    "geo_spatial_grid_datasets",
    "r{}_empty_cells_with_elevation_polars.tf".format(params["h3_resolution"]),
)

In [ ]:
total = len(combined_with_elevation_polars)
print("  writing tfrecords")
with tf.io.TFRecordWriter(tfrecord_file) as writer:
    i = 0
    for i in tqdm(range(len(combined_with_elevation_polars))):
        if i % 100_000 == 0:
            print("  {} of {} written".format(i, total))

        row_polars = combined_with_elevation_polars[i]
        row_pandas = combined_with_elevation_pandas.iloc[i]
        
        lat_pandas = row_pandas["lat"]
        lng_pandas = row_pandas["lng"]
        norm_lat_pandas = lat_pandas / 90.0
        norm_lng_pandas = lng_pandas / 180.0
        norm_loc_pandas = np.array([norm_lng_pandas, norm_lat_pandas])
        encoded_loc_pandas = np.concatenate(
            [np.sin(norm_loc_pandas * math.pi), np.cos(norm_loc_pandas * math.pi)]
        )


        l0 = encoded_loc_pandas[0]
        l1 = encoded_loc_pandas[1]
        l2 = encoded_loc_pandas[2]
        l3 = encoded_loc_pandas[3]

        elevation = row_pandas["elevation"]
        cids = sorted(list(row_pandas["spatial_class_id"]))
        
        example_pandas = create_tf_example(l0, l1, l2, l3, elevation, cids)

        l0 = row_polars["a"].item()
        l1 = row_polars["b"].item()
        l2 = row_polars["c"].item()
        l3 = row_polars["d"].item()

        elevation = row_polars["elevation"].item()
        cids = sorted(list(row_polars["spatial_class_id"].item()))
        
        example_polars = create_tf_example(l0, l1, l2, l3, elevation, cids)
        if (example_pandas != example_polars):
            print(example_pandas)
            print(example_polars)

        
        writer.write(example_polars.SerializeToString())
        i += 1
    writer.close()


In [ ]:
39.868014 / 180.0